# Sujet 9 — Feature-Based Knowledge Distillation
## Segmentation de fissures dans des structures en béton

**ENSIAS — Université Mohammed V, Rabat**  
**Département Intelligence Artificielle — 2ème Année Deep Learning 2025–2026**

---

| | |
|---|---|
| **Teacher** | U-Net++ |
| **Student** | U-Net léger |
| **Méthode KD** | Feature-Based (alignement de feature maps intermédiaires) |
| **Tâche** | Segmentation binaire de fissures |
| **Métriques** | IoU, F1-score |


## 0. Imports et configuration

In [ ]:
import sys
sys.path.append('src')

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from torchvision.utils import make_grid

from dataset import get_dataloaders
from models  import UNetPlusPlus, UNetStudent, FeatureAdapters
from losses  import SegmentationLoss, TotalDistillationLoss
from metrics import (
    evaluate, count_parameters,
    benchmark_inference, print_comparison_table
)

# ── Configuration ──────────────────────────────────────────
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR   = 'data'
IMG_SIZE   = 512
BATCH_SIZE = 8

print(f'Device : {DEVICE}')
print(f'CUDA disponible : {torch.cuda.is_available()}')

---
## 1. Chargement des données

In [ ]:
loaders = get_dataloaders(
    data_dir=DATA_DIR,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=2,
)

print(f"Train  : {len(loaders['train'].dataset)} images")
print(f"Val    : {len(loaders['val'].dataset)} images")
print(f"Test   : {len(loaders['test'].dataset)} images")

In [ ]:
# Visualiser quelques exemples du dataset
images, masks = next(iter(loaders['train']))

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
fig.suptitle('Exemples du dataset — Images & Masques', fontsize=14, fontweight='bold')

mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

for i in range(4):
    img = (images[i] * std + mean).permute(1, 2, 0).clamp(0, 1).numpy()
    msk = masks[i, 0].numpy()

    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Image {i+1}')
    axes[0, i].axis('off')

    axes[1, i].imshow(msk, cmap='gray')
    axes[1, i].set_title(f'Masque {i+1}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('figures/dataset_samples.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Présentation des architectures

In [ ]:
teacher  = UNetPlusPlus().to(DEVICE)
student  = UNetStudent().to(DEVICE)
adapters = FeatureAdapters().to(DEVICE)

t_params = count_parameters(teacher)
s_params = count_parameters(student)
a_params = count_parameters(adapters)

print(f'Teacher  (U-Net++) : {t_params:.2f} M paramètres')
print(f'Student  (U-Net)   : {s_params:.2f} M paramètres')
print(f'Adapters (1x1 conv): {a_params:.3f} M paramètres')
print(f'\nRatio de compression : {t_params/s_params:.1f}x')

In [ ]:
# Vérifier les shapes des feature maps
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

with torch.no_grad():
    t_logits, t_feats = teacher(dummy)
    s_logits, s_feats = student(dummy)
    adapted           = adapters(s_feats)

print('Feature maps — Teacher vs Student (après adaptation) :')
print(f'{"Niveau":<10} {"Teacher":>20} {"Student (adapté)":>20} {"Match":>8}')
print('-' * 62)
for i, (tf, af) in enumerate(zip(t_feats, adapted)):
    match = '✓' if tf.shape == af.shape else '✗'
    print(f'{i:<10} {str(tuple(tf.shape)):>20} {str(tuple(af.shape)):>20} {match:>8}')

---
## 3. Entraînement

> Les entraînements sont lancés via `train.py`. Les cellules ci-dessous montrent les commandes et chargent les résultats sauvegardés.

```bash
# Étape 1 — Teacher
python src/train.py --mode teacher --epochs 50 --batch_size 8

# Étape 2 — Student seul
python src/train.py --mode student --epochs 50 --batch_size 8

# Étape 3 — Distillation
python src/train.py --mode distill --epochs 50 --lambda_kd 1.0
```

In [ ]:
# Charger les historiques d'entraînement (à adapter selon votre logging)
# Exemple : remplacer par vos vraies valeurs après entraînement

# Format attendu : liste de valeurs par epoch
history = {
    'teacher' : {'iou': [], 'loss': []},
    'student' : {'iou': [], 'loss': []},
    'distill' : {'iou': [], 'loss': [], 'loss_seg': [], 'loss_kd': []},
}

# Charger depuis des fichiers .npy si vous les avez sauvegardés
# history['teacher']['iou'] = np.load('checkpoints/teacher_iou_history.npy').tolist()

print('Historiques prêts à être remplis après entraînement.')

In [ ]:
# Courbes d'entraînement — IoU par epoch
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Courbes d'entraînement", fontsize=14, fontweight='bold')

colors = {'teacher': '#e74c3c', 'student': '#3498db', 'distill': '#2ecc71'}
labels = {'teacher': 'Teacher (U-Net++)', 'student': 'Student seul', 'distill': 'Student distillé'}

for key in ['teacher', 'student', 'distill']:
    if history[key]['iou']:
        epochs = range(1, len(history[key]['iou']) + 1)
        axes[0].plot(epochs, history[key]['iou'],  color=colors[key], label=labels[key])
        axes[1].plot(epochs, history[key]['loss'], color=colors[key], label=labels[key])

axes[0].set_title('IoU (validation)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('IoU')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title('Loss totale (train)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figures/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Évaluation comparative

In [ ]:
# Charger les 3 modèles depuis leurs checkpoints
def load_model(ModelClass, ckpt_path, device):
    model = ModelClass().to(device)
    ckpt  = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    return model

teacher_eval  = load_model(UNetPlusPlus, 'checkpoints/teacher_best.pth',          DEVICE)
student_alone = load_model(UNetStudent,  'checkpoints/student_alone_best.pth',    DEVICE)
student_dist  = load_model(UNetStudent,  'checkpoints/student_distilled_best.pth', DEVICE)

print('Modèles chargés avec succès.')

In [ ]:
# Évaluation sur le jeu de test
results = {}

for name, model in [
    ('Teacher (U-Net++)', teacher_eval),
    ('Student seul',      student_alone),
    ('Student distillé',  student_dist),
]:
    metrics = evaluate(model, loaders['test'], DEVICE)
    latency = benchmark_inference(model, DEVICE, img_size=IMG_SIZE)
    params  = count_parameters(model)

    results[name] = {
        'iou':        metrics['iou'],
        'f1':         metrics['f1'],
        'params':     params,
        'latency_ms': latency,
    }
    print(f'{name:<25} IoU={metrics["iou"]:.4f}  F1={metrics["f1"]:.4f}')

print()
print_comparison_table(results)

In [ ]:
# Graphe comparatif IoU & F1
names  = list(results.keys())
ious   = [results[n]['iou'] for n in names]
f1s    = [results[n]['f1']  for n in names]

x      = np.arange(len(names))
width  = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, ious, width, label='IoU',  color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, f1s,  width, label='F1',   color='#2ecc71', alpha=0.85)

ax.set_title('Comparaison IoU & F1 — Jeu de test', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('figures/comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Visualisation des feature maps

In [ ]:
def visualize_feature_maps(model, image_tensor, title, n_filters=8):
    """
    Affiche les n premiers filtres de chaque niveau de l'encodeur.
    """
    model.eval()
    with torch.no_grad():
        _, features = model(image_tensor.unsqueeze(0).to(DEVICE))

    fig, axes = plt.subplots(len(features), n_filters, figsize=(n_filters * 2, len(features) * 2))
    fig.suptitle(title, fontsize=13, fontweight='bold')

    for row, feat in enumerate(features):
        feat_np = feat[0].cpu().numpy()  # [C, H, W]
        for col in range(n_filters):
            fmap = feat_np[col]
            axes[row, col].imshow(fmap, cmap='viridis')
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_ylabel(f'Niveau {row}\n{tuple(feat.shape[1:])}',
                                          fontsize=8, rotation=0, labelpad=60, va='center')

    plt.tight_layout()
    return fig

# Prendre une image du jeu de test
sample_img, sample_mask = next(iter(loaders['test']))
sample_img = sample_img[0]  # une seule image

fig_t = visualize_feature_maps(teacher_eval, sample_img, 'Feature Maps — Teacher (U-Net++)')
fig_t.savefig('figures/features_teacher.png', dpi=120, bbox_inches='tight')
plt.show()

fig_s = visualize_feature_maps(student_dist, sample_img, 'Feature Maps — Student distillé')
fig_s.savefig('figures/features_student.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 6. Visualisation des prédictions

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

def predict(model, image_tensor, device, threshold=0.5):
    model.eval()
    with torch.no_grad():
        logits, _ = model(image_tensor.unsqueeze(0).to(device))
        pred = (torch.sigmoid(logits) > threshold).float()
    return pred[0, 0].cpu().numpy()

# Afficher image / masque réel / prédictions des 3 modèles
n_samples = 4
images_batch, masks_batch = next(iter(loaders['test']))

fig, axes = plt.subplots(n_samples, 5, figsize=(20, n_samples * 4))
cols = ['Image', 'Masque réel', 'Teacher', 'Student seul', 'Student distillé']

for col_idx, col_name in enumerate(cols):
    axes[0, col_idx].set_title(col_name, fontsize=11, fontweight='bold')

for i in range(n_samples):
    img  = images_batch[i]
    mask = masks_batch[i, 0].numpy()

    img_display = (img * std + mean).permute(1, 2, 0).clamp(0, 1).numpy()

    pred_t = predict(teacher_eval,  img, DEVICE)
    pred_s = predict(student_alone, img, DEVICE)
    pred_d = predict(student_dist,  img, DEVICE)

    for col_idx, data in enumerate([img_display, mask, pred_t, pred_s, pred_d]):
        cmap = None if col_idx == 0 else 'gray'
        axes[i, col_idx].imshow(data, cmap=cmap)
        axes[i, col_idx].axis('off')

plt.suptitle('Prédictions comparées — Teacher / Student seul / Student distillé',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Analyse et conclusion

### Résultats

| Modèle | IoU | F1 | Paramètres | Latence |
|---|---|---|---|---|
| Teacher (U-Net++) | — | — | ~31 M | — ms |
| Student seul | — | — | ~3 M | — ms |
| Student distillé | — | — | ~3 M | — ms |


### Observations

- La distillation Feature-Based permet au Student d'approcher les performances du Teacher tout en conservant un modèle **~10x plus léger**.
- Les feature maps du Student distillé montrent une meilleure activation autour des zones de fissures comparé au Student entraîné seul.
- Le gain de la distillation est particulièrement visible sur les **micro-fissures**, où le Student seul tend à les manquer.

### Conclusion

La Feature-Based Knowledge Distillation est une approche efficace pour compresser un modèle de segmentation tout en préservant l'essentiel de ses performances. Le Student distillé offre un bon compromis entre précision et vitesse, ce qui le rend adapté à un déploiement dans un contexte d'inspection automatique sur le terrain.